In [1]:
import pandas as pd
from pathlib import Path

from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [2]:
current_path = Path.cwd()

if current_path.name == "notebooks":
    project_root = current_path.parent
else:
    project_root = current_path

cleaned_file = project_root / "data" / "processed" / "student_dropout_cleaned.csv"

df = pd.read_csv(cleaned_file)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (3630, 37)


,marital_status,application_mode,application_order,course,daytime_evening_attendance,previous_qualification,previous_qualification_grade,nacionality,mothers_qualification,fathers_qualification,...,curricular_units_2nd_sem_credited,curricular_units_2nd_sem_enrolled,curricular_units_2nd_sem_evaluations,curricular_units_2nd_sem_approved,curricular_units_2nd_sem_grade,curricular_units_2nd_sem_without_evaluations,unemployment_rate,inflation_rate,gdp,dropout_risk
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,1
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,0
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,1
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,0
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,0


In [3]:
second_sem_features = [col for col in df.columns if "2nd_sem" in col]

print("Second-semester features to remove:")
print(second_sem_features)

df_first_sem = df.drop(columns=second_sem_features)

print("First-semester dataset shape:", df_first_sem.shape)

Second-semester features to remove:
['curricular_units_2nd_sem_credited', 'curricular_units_2nd_sem_enrolled', 'curricular_units_2nd_sem_evaluations', 'curricular_units_2nd_sem_approved', 'curricular_units_2nd_sem_grade', 'curricular_units_2nd_sem_without_evaluations']
First-semester dataset shape: (3630, 31)


In [4]:
X = df_first_sem.drop(columns=["dropout_risk"])
y = df_first_sem["dropout_risk"]

print("Features:", X.shape)
print("Target:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())
print(y.value_counts(normalize=True) * 100)

Features: (3630, 30)
Target: (3630,)

Target distribution:
dropout_risk
0    2209
1    1421
Name: count, dtype: int64
dropout_risk
0    60.853994
1    39.146006
Name: proportion, dtype: float64


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (2904, 30)
X_test: (726, 30)
y_train: (2904,)
y_test: (726,)


In [6]:
xgb_first_sem_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

xgb_first_sem_model.fit(X_train, y_train)

print("First-semester XGBoost model trained successfully.")

First-semester XGBoost model trained successfully.


In [7]:
y_pred = xgb_first_sem_model.predict(X_test)
y_proba = xgb_first_sem_model.predict_proba(X_test)[:, 1]

print("Predictions completed.")

Predictions completed.


In [8]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

first_sem_metrics = pd.DataFrame({
    "Model": ["XGBoost First-Semester"],
    "Accuracy": [accuracy],
    "Precision": [precision],
    "Recall": [recall],
    "F1-score": [f1],
    "ROC-AUC": [roc_auc]
})

first_sem_metrics

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,XGBoost First-Semester,0.902204,0.884477,0.862676,0.87344,0.953429


In [9]:
print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.93      0.92       442
           1       0.88      0.86      0.87       284

    accuracy                           0.90       726
   macro avg       0.90      0.90      0.90       726
weighted avg       0.90      0.90      0.90       726

Confusion Matrix:
[[410  32]
 [ 39 245]]


In [10]:
results_path = project_root / "results"
models_path = project_root / "models"

results_path.mkdir(parents=True, exist_ok=True)
models_path.mkdir(parents=True, exist_ok=True)

first_sem_metrics.to_csv(results_path / "xgboost_first_semester_metrics.csv", index=False)

xgb_first_sem_model.save_model(models_path / "xgboost_first_semester_model.json")

print("Metrics saved to:", results_path / "xgboost_first_semester_metrics.csv")
print("Model saved to:", models_path / "xgboost_first_semester_model.json")

Metrics saved to: c:\Users\sandy\Documents\git\student-dropout-xai-pipeline\results\xgboost_first_semester_metrics.csv
Model saved to: c:\Users\sandy\Documents\git\student-dropout-xai-pipeline\models\xgboost_first_semester_model.json
